# Sample Final Project: Security Metrics Daily Summary

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST2412_Data_Security_Privacy_Ethics/blob/main/final_project/examples/sample_security_metrics_project.ipynb)

**Important:** This is an inspiration sample, not language to copy into your own final project.

This notebook shows one possible beginner coding project for CST 2412. The goal is to use a small security metrics CSV to decide which systems deserve review first, then explain the result in plain language.


## What This Notebook Demonstrates

This example uses beginner Python and pandas to:

- load a CSV file from GitHub
- inspect the table
- calculate a simple priority score
- sort systems by priority
- make one simple chart
- produce a summary table
- connect code results to a short analyst-style final project report


In [ ]:

# pandas is the main Python library we use for working with tables.
# Most CSV files become a pandas DataFrame.
# A DataFrame is like a spreadsheet: it has rows, columns, and values.
import pandas as pd

# matplotlib is a common Python plotting library.
# pyplot is the part of matplotlib we use for simple charts.
import matplotlib.pyplot as plt


## Step 1: Load the CSV From GitHub

The file lives in the course GitHub repository. A **raw GitHub URL** gives pandas direct access to the CSV contents.


In [ ]:

# This is the raw URL for the Week 12 security metrics CSV.
# It points directly to the CSV data, not to the normal GitHub web page.
csv_url = "https://raw.githubusercontent.com/lolusername/CST2412_Data_Security_Privacy_Ethics/main/week_12/day_1/data/week12_security_metrics.csv"

# pd.read_csv(...) loads a CSV file into a pandas DataFrame.
# The variable name metrics is our name for the table after it loads.
metrics = pd.read_csv(csv_url)

# .head() shows the first five rows.
# This is usually the first thing to do after loading a dataset.
metrics.head()


## Step 2: Inspect the Table

Before analyzing a dataset, check the columns and the basic shape. This helps avoid guessing what the data means.


In [ ]:

# .shape tells us the size of the table.
# The first number is rows. The second number is columns.
metrics.shape


In [ ]:

# .columns shows the column names.
# Column names tell us what fields are available for analysis.
metrics.columns


In [ ]:

# .info() shows each column, how many non-empty values it has, and its data type.
# object usually means text.
# int64 usually means whole numbers.
metrics.info()


## Step 3: Create Helper Functions

The priority score is intentionally simple. It is not a professional detection model. It is a beginner-friendly way to practice turning signals into a ranked list.


In [ ]:

def yes_no_points(value):
    # This function converts yes/no text into points.
    # str(value) protects us in case the value is not already text.
    # .strip() removes extra spaces.
    # .lower() makes YES, Yes, and yes all compare the same way.
    if str(value).strip().lower() == "yes":
        return 1
    return 0


def severity_points(value):
    # This function converts alert severity into points.
    # High severity is worth 2 points, medium is worth 1, and low is worth 0.
    severity = str(value).strip().lower()
    if severity == "high":
        return 2
    if severity == "medium":
        return 1
    return 0


def criticality_points(value):
    # This function converts business criticality into points.
    # A high-criticality system matters more because harm to that system matters more.
    criticality = str(value).strip().lower()
    if criticality == "high":
        return 2
    if criticality == "medium":
        return 1
    return 0


## Step 4: Build a Priority Score

This score combines security signals and business context. That matters because alerts alone do not tell the whole story.


In [ ]:

def build_priority_score(row):
    # This function calculates one priority score for one row.
    # In pandas, one row is represented as a Series.
    # A Series is like a labeled list of values from the same row.
    score = 0

    # Alert severity is a signal that something may already be wrong.
    score += severity_points(row["highest_alert_severity"])

    # Asset criticality is context about how important the system is.
    score += criticality_points(row["asset_criticality"])

    # Internet exposure adds risk because the system is reachable from outside.
    score += yes_no_points(row["internet_exposed"])

    # Sensitive data adds risk because the harm could be higher if exposed.
    score += yes_no_points(row["sensitive_data"])

    # Many failed logins can suggest password guessing or password spraying.
    if row["failed_logins_24h"] >= 30:
        score += 1

    # A successful login after failures can be more serious than failures alone.
    if row["successful_logins_after_failures"] > 0:
        score += 1

    # Many MFA denials can suggest someone is trying to get past account protection.
    if row["mfa_denials"] >= 5:
        score += 1

    # Malware alerts suggest the system may need direct review.
    if row["malware_alerts"] > 0:
        score += 1

    # The function returns the final number.
    return score


In [ ]:

# .apply(..., axis=1) runs the function once for each row.
# axis=1 means pandas passes rows into the function.
# The result is saved into a new column called priority_score.
metrics["priority_score"] = metrics.apply(build_priority_score, axis=1)

# Show the first few rows so we can confirm the new column exists.
metrics.head()


## Step 5: Sort Systems by Priority

Sorting helps us decide what to review first. This does not prove an incident happened. It only creates a reasonable first-review order.


In [ ]:

# sort_values sorts the table by a column.
# ascending=False means highest scores appear first.
ranked = metrics.sort_values("priority_score", ascending=False)

# Display the systems from highest to lowest priority.
ranked[["asset_id", "system_name", "business_unit", "highest_alert_severity", "priority_score", "notes"]]


## Step 6: Make One Simple Chart

A chart is optional for the final project, but one simple chart can make a report easier to read.


In [ ]:

# Create a small table with only the columns needed for the chart.
chart_data = ranked[["system_name", "priority_score"]]

# figsize controls the chart size.
plt.figure(figsize=(10, 5))

# plt.bar creates a bar chart.
# The first argument is the labels on the x-axis.
# The second argument is the numeric values on the y-axis.
plt.bar(chart_data["system_name"], chart_data["priority_score"])

# Rotate labels so long system names are easier to read.
plt.xticks(rotation=45, ha="right")

# Add a useful title and axis labels.
plt.title("Simple Priority Score by System")
plt.xlabel("System")
plt.ylabel("Priority Score")

# tight_layout reduces cut-off labels.
plt.tight_layout()

# Show the chart.
plt.show()


## Step 7: Create a Summary Table

A final project report should not include every column if that makes the result hard to read. Choose the columns that support your explanation.


In [ ]:

# This list controls which columns appear in the final summary table.
summary_columns = [
    "asset_id",
    "system_name",
    "business_unit",
    "asset_criticality",
    "sensitive_data",
    "internet_exposed",
    "highest_alert_severity",
    "priority_score",
    "notes",
]

# ranked[summary_columns] keeps only those columns.
summary = ranked[summary_columns]

# Display the summary table.
summary


In [ ]:

# Optional: save the summary table as a CSV.
# index=False prevents pandas from writing the row number as an extra column.
summary.to_csv("sample_security_metrics_summary.csv", index=False)


## Step 8: Turn the Code Into a Final Project Report

A coding final project is not just code. The report explains what the code means.

A report based on this notebook could include:

- project question: Which system should a junior analyst review first?
- dataset description: each row is one fictional system from a daily security metrics export
- method: calculate a simple priority score from alert severity, criticality, exposure, sensitive data, and login signals
- findings: highest priority systems combine technical signals with business context
- recommendation: review the highest-scoring sensitive or critical systems first
- privacy concern: security logs should be limited to security purposes and not reused for unrelated surveillance
- limitation: a score does not prove an incident happened
